# DeepGuard — Optimized High-Performance Model Training Pipeline

This optimized notebook trains baseline architectures on preprocessed cropped face frames.

**Performance & Stability Features:**
- **Fast SSD Caching**: Automatically syncs dataset files to local `/content/` NVMe memory to bypass Google Drive latency bottlenecks.
- **Binary Balance Assurance**: Verifies both REAL and FAKE classes are present across all splits.
- **Pinned Dependencies**: Resolves Colab PyTorch/Pandas version conflicts.
- **Optimized PyTorch Execution**: Enables `tf32` precision, non-blocking GPU memory copies, and `persistent_workers`.

In [ ]:
# Install compatible dependencies without breaking Colab environment
!pip install -q "Pillow>=10.0.0,<11.0.0" "torchvision>=0.16.0" scikit-learn "pandas>=2.0.0,<2.3.0" matplotlib seaborn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 42.2 MB/s eta 0:00:00


In [ ]:
import os
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, roc_auc_score, precision_recall_fscore_support

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# Optimize PyTorch GPU Compute (Ampere / Ada / Hopper / Turing optimizations)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__} | Active Device: {device}")
if torch.cuda.is_available():
    print("GPU Model:", torch.cuda.get_device_name(0))

Mounted at /content/drive
PyTorch Version: 2.11.0+cu128 | Active Device: cuda
GPU Model: Tesla T4


In [ ]:
RANDOM_SEED = 42
BATCH_SIZE = 64  # Increased for faster batch throughput on T4/A100 GPUs
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
IMAGE_SIZE = 224
NUM_WORKERS = 2

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Setup Directory Paths
PROJECT_DIR = Path("/content/drive/MyDrive/DeepGuard")
DATA_DIR = PROJECT_DIR / "dataset_analysis"
LOG_DIR = DATA_DIR / "preprocessing_logs"
MODEL_SAVE_DIR = PROJECT_DIR / "trained_models"
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

FACES_METADATA_FILE = LOG_DIR / "processed_faces_metadata.csv"
if not FACES_METADATA_FILE.exists():
    raise FileNotFoundError(f"{FACES_METADATA_FILE} missing! Please complete preprocessing step first.")

df_faces = pd.read_csv(FACES_METADATA_FILE)

# --- VALIDATE DATASET CLASS BALANCE ---
print(f"Loaded {len(df_faces)} total face samples from metadata.")
split_counts = df_faces.groupby(["split", "label_name"]).size()
print("\nDataset Breakdown:")
print(split_counts)

unique_labels = df_faces["label_name"].unique()
if len(unique_labels) < 2:
    raise ValueError(
        f"CRITICAL ERROR: Dataset contains only 1 class: {unique_labels}. "
        "You must re-run preprocessing to include BOTH 'REAL' and 'FAKE' samples before training!"
    )
else:
    print("\nDataset contains both REAL and FAKE classes. Proceeding...")

Loaded 2929 total face samples from metadata.

Dataset Breakdown:
split  label_name
test   FAKE           249
       REAL           250
train  FAKE           932
       REAL          1000
val    FAKE           248
       REAL           250
dtype: int64

✅ Dataset contains both REAL and FAKE classes. Proceeding...


In [ ]:
# PyTorch Dataset & Fast Memory Loader
class DeepfakeFaceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.paths = self.df["image_path"].tolist()
        self.labels = self.df["label"].astype(np.float32).tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        label = self.labels[idx]

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

# Optimized Transformations
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_df = df_faces[df_faces["split"] == "train"]
val_df = df_faces[df_faces["split"] == "val"]
test_df = df_faces[df_faces["split"] == "test"]

# Optimized DataLoaders using persistent workers and pin memory
train_loader = DataLoader(
    DeepfakeFaceDataset(train_df, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True if NUM_WORKERS > 0 else False
)
val_loader = DataLoader(
    DeepfakeFaceDataset(val_df, val_test_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True if NUM_WORKERS > 0 else False
)
test_loader = DataLoader(
    DeepfakeFaceDataset(test_df, val_test_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"DataLoaders Initialized: Train ({len(train_loader.dataset)}), Val ({len(val_loader.dataset)}), Test ({len(test_loader.dataset)})")

DataLoaders Initialized: Train (1932), Val (498), Test (499)


In [ ]:
def build_model(architecture_name):
    """Factory to initialize models with binary classification heads."""
    if architecture_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, 1)

    elif architecture_name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, 1)

    elif architecture_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, 1)

    else:
        raise ValueError(f"Unsupported architecture: {architecture_name}")

    return model.to(device)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = 0.0
    preds_all, labels_all = [], []
    amp_device_type = "cuda" if device.type == "cuda" else "cpu"

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).unsqueeze(1).float()
        optimizer.zero_grad(set_to_none=True)  # Faster zeroing

        with torch.amp.autocast(amp_device_type, enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(outputs).detach().cpu().numpy().flatten()
        preds_all.extend(probs)
        labels_all.extend(labels.cpu().numpy().flatten())

    epoch_loss = total_loss / len(loader.dataset)
    preds_binary = (np.array(preds_all) >= 0.5).astype(int)
    epoch_acc = accuracy_score(labels_all, preds_binary)
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    probs_all, labels_all = [], []
    amp_device_type = "cuda" if device.type == "cuda" else "cpu"

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).unsqueeze(1).float()

        with torch.amp.autocast(amp_device_type, enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(outputs).cpu().numpy().flatten()
        probs_all.extend(probs)
        labels_all.extend(labels.cpu().numpy().flatten())

    eval_loss = total_loss / len(loader.dataset)
    probs_all = np.array(probs_all)
    labels_all = np.array(labels_all)
    preds_binary = (probs_all >= 0.5).astype(int)

    acc = accuracy_score(labels_all, preds_binary)
    auc = roc_auc_score(labels_all, probs_all) if len(np.unique(labels_all)) > 1 else 0.5
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_all, preds_binary, average="binary", zero_division=0
    )

    return {
        "loss": eval_loss,
        "accuracy": acc,
        "auc": auc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "probs": probs_all,
        "labels": labels_all
    }

In [ ]:
architectures = ["mobilenet_v2", "resnet50", "efficientnet_b0"]
model_results = {}

for arch in architectures:
    print("\n" + "=" * 50)
    print(f"Starting Training: {arch.upper()}")
    print("=" * 50)

    model = build_model(arch)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    best_val_loss = float("inf")
    best_model_path = MODEL_SAVE_DIR / f"{arch}_best.pth"
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_auc": []}

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scaler)
        val_metrics = evaluate(model, val_loader, criterion)
        scheduler.step(val_metrics["loss"])
        elapsed = time.time() - t0

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["accuracy"])
        history["val_auc"].append(val_metrics["auc"])

        print(
            f"Epoch {epoch:02d}/{NUM_EPOCHS:02d} [{elapsed:.1f}s] | "
            f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} Acc: {val_metrics['accuracy']:.4f} AUC: {val_metrics['auc']:.4f}"
        )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            torch.save(model.state_dict(), best_model_path)

    if best_model_path.exists():
        model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))

    test_metrics = evaluate(model, test_loader, criterion)
    model_results[arch] = {"history": history, "test_metrics": test_metrics}

    print(f"\n>>> Test Set Performance [{arch.upper()}]:")
    print(
        f"Accuracy: {test_metrics['accuracy']:.4f} | ROC-AUC: {test_metrics['auc']:.4f} | "
        f"Precision: {test_metrics['precision']:.4f} | Recall: {test_metrics['recall']:.4f} | F1: {test_metrics['f1']:.4f}"
    )


Starting Training: MOBILENET_V2
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 48.8MB/s]


Epoch 01/10 [402.7s] | Train Loss: 0.5993 Acc: 0.7479 | Val Loss: 0.5564 Acc: 0.7189 AUC: 0.8148
Epoch 02/10 [14.2s] | Train Loss: 0.3271 Acc: 0.8841 | Val Loss: 0.4022 Acc: 0.8434 AUC: 0.9316
Epoch 03/10 [13.8s] | Train Loss: 0.1617 Acc: 0.9467 | Val Loss: 0.4726 Acc: 0.7851 AUC: 0.9298
Epoch 04/10 [14.2s] | Train Loss: 0.0867 Acc: 0.9788 | Val Loss: 0.2918 Acc: 0.8775 AUC: 0.9672
Epoch 05/10 [14.2s] | Train Loss: 0.0509 Acc: 0.9881 | Val Loss: 0.3804 Acc: 0.8414 AUC: 0.9458
Epoch 06/10 [13.7s] | Train Loss: 0.0366 Acc: 0.9912 | Val Loss: 0.3257 Acc: 0.8775 AUC: 0.9526
Epoch 07/10 [14.1s] | Train Loss: 0.0320 Acc: 0.9943 | Val Loss: 0.2910 Acc: 0.8855 AUC: 0.9568
Epoch 08/10 [14.2s] | Train Loss: 0.0232 Acc: 0.9948 | Val Loss: 0.3431 Acc: 0.8655 AUC: 0.9467
Epoch 09/10 [13.6s] | Train Loss: 0.0230 Acc: 0.9948 | Val Loss: 0.3219 Acc: 0.8855 AUC: 0.9534
Epoch 10/10 [12.9s] | Train Loss: 0.0201 Acc: 0.9948 | Val Loss: 0.3500 Acc: 0.8755 AUC: 0.9460

>>> Test Set Performance [MOBILENET_V2

100%|██████████| 97.8M/97.8M [00:00<00:00, 170MB/s]


Epoch 01/10 [39.2s] | Train Loss: 0.5405 Acc: 0.7697 | Val Loss: 0.3919 Acc: 0.7912 AUC: 0.9243
Epoch 02/10 [17.4s] | Train Loss: 0.1500 Acc: 0.9503 | Val Loss: 0.6073 Acc: 0.7590 AUC: 0.9546
Epoch 03/10 [15.6s] | Train Loss: 0.0504 Acc: 0.9845 | Val Loss: 0.3516 Acc: 0.8614 AUC: 0.9643
Epoch 04/10 [18.8s] | Train Loss: 0.0294 Acc: 0.9928 | Val Loss: 0.2232 Acc: 0.9116 AUC: 0.9722
Epoch 05/10 [17.5s] | Train Loss: 0.0182 Acc: 0.9943 | Val Loss: 0.4866 Acc: 0.8675 AUC: 0.9344
Epoch 06/10 [15.4s] | Train Loss: 0.0254 Acc: 0.9917 | Val Loss: 0.3693 Acc: 0.8594 AUC: 0.9562
Epoch 07/10 [15.5s] | Train Loss: 0.0183 Acc: 0.9943 | Val Loss: 0.3038 Acc: 0.9016 AUC: 0.9639
Epoch 08/10 [15.7s] | Train Loss: 0.0144 Acc: 0.9964 | Val Loss: 0.3172 Acc: 0.8855 AUC: 0.9672
Epoch 09/10 [15.9s] | Train Loss: 0.0092 Acc: 0.9984 | Val Loss: 0.4410 Acc: 0.8454 AUC: 0.9443
Epoch 10/10 [15.4s] | Train Loss: 0.0125 Acc: 0.9969 | Val Loss: 0.2969 Acc: 0.9036 AUC: 0.9631

>>> Test Set Performance [RESNET50]:
Ac

100%|██████████| 20.5M/20.5M [00:00<00:00, 117MB/s] 


Epoch 01/10 [100.1s] | Train Loss: 0.5647 Acc: 0.7774 | Val Loss: 0.4734 Acc: 0.7851 AUC: 0.8963
Epoch 02/10 [14.8s] | Train Loss: 0.2376 Acc: 0.9317 | Val Loss: 0.6266 Acc: 0.7570 AUC: 0.9157
Epoch 03/10 [14.2s] | Train Loss: 0.0970 Acc: 0.9767 | Val Loss: 0.3743 Acc: 0.8574 AUC: 0.9483
Epoch 04/10 [14.8s] | Train Loss: 0.0452 Acc: 0.9902 | Val Loss: 0.4080 Acc: 0.8574 AUC: 0.9307
Epoch 05/10 [14.2s] | Train Loss: 0.0347 Acc: 0.9912 | Val Loss: 0.4445 Acc: 0.8554 AUC: 0.9373
Epoch 06/10 [14.4s] | Train Loss: 0.0208 Acc: 0.9979 | Val Loss: 0.4698 Acc: 0.8494 AUC: 0.9429
Epoch 07/10 [14.3s] | Train Loss: 0.0144 Acc: 0.9964 | Val Loss: 0.5246 Acc: 0.8353 AUC: 0.9407
Epoch 08/10 [14.1s] | Train Loss: 0.0171 Acc: 0.9938 | Val Loss: 0.4465 Acc: 0.8534 AUC: 0.9420
Epoch 09/10 [14.2s] | Train Loss: 0.0199 Acc: 0.9933 | Val Loss: 0.4705 Acc: 0.8353 AUC: 0.9393
Epoch 10/10 [14.2s] | Train Loss: 0.0141 Acc: 0.9969 | Val Loss: 0.4775 Acc: 0.8514 AUC: 0.9464

>>> Test Set Performance [EFFICIENTNET

In [ ]:
# Benchmark Summary Export
summary_data = []
for arch, res in model_results.items():
    tm = res["test_metrics"]
    summary_data.append({
        "Model": arch,
        "Test Accuracy": tm["accuracy"],
        "Test ROC-AUC": tm["auc"],
        "Test Precision": tm["precision"],
        "Test Recall": tm["recall"],
        "Test F1-Score": tm["f1"]
    })

df_summary = pd.DataFrame(summary_data)
df_summary.to_csv(LOG_DIR / "model_comparison_benchmark.csv", index=False)
print("\n=== FINAL MODEL PERFORMANCE BENCHMARK ===")
print(df_summary.to_string(index=False))


=== FINAL MODEL PERFORMANCE BENCHMARK ===
          Model  Test Accuracy  Test ROC-AUC  Test Precision  Test Recall  Test F1-Score
   mobilenet_v2       0.873747      0.940988        0.823529        0.952       0.883117
       resnet50       0.895792      0.963703        0.919492        0.868       0.893004
efficientnet_b0       0.843687      0.948498        0.784768        0.948       0.858696
